# Large-scale multivariate Gaussian process emulation of joint contact forces

A trivariate separable space-time Matérn emulator of the hip, knee and ankle joint contact forces, fitted by maximum likelihood on one GPU with an adaptive block mixed-precision Cholesky factorization.

Joint contact forces are the resultant forces transmitted across a joint. They are estimated from gait laboratory measurements by musculoskeletal modeling and simulation in OpenSim, a pipeline that takes hours for the walking data of a group of participants. A Gaussian process (GP) emulator is a statistical model trained on the OpenSim results that predicts the joint contact force curves of a new stride from its measured inputs, with a predictive distribution that quantifies its uncertainty. The first notebook of this project, `jcf-gp-colab.ipynb`, builds a univariate emulator, one joint at a time, with a covariance function that is separable in the inputs and in gait time. Separability gives the covariance matrix a Kronecker structure, so that a matrix with 48,480 rows never has to be formed and each fit takes well under a minute.

That speed is bought with two modeling restrictions. The first is that each joint is modeled alone, although the hip, knee and ankle contact forces of one stride are produced by the same ground reaction force and by muscles that cross more than one joint, so that their deviations from the mean curves are correlated. The second appears as soon as the three joints are modeled jointly within the Kronecker structure, because that structure requires the three joints to share one input range and one temporal range, whereas the univariate fits estimate a different pair of ranges for each joint. This notebook removes both restrictions with a trivariate separable space-time Matérn model. Each joint keeps its own variance, input range, temporal range and nugget effect, the covariance function of every joint remains the separable product of a Matérn covariance function in the inputs and a Matérn covariance function in gait time, and every pair of joints has a cross-correlation. The three univariate emulators and the equal-range Kronecker model are special cases of this model, which means that the likelihood measures how much the cross-correlations and the joint-specific ranges each contribute.

The price of the better model is computational. With joint-specific ranges the covariance matrix is no longer a Kronecker product, and it has to be treated as a dense matrix with $N = 3 \times 480 \times 101 = 145{,}440$ rows. Maximum likelihood estimation requires the Cholesky factorization of this matrix at every evaluation of the likelihood, which costs $O(N^3)$ floating-point operations and $O(N^2)$ memory. In double precision the matrix occupies 169 GB and its Cholesky factor 85 GB, while the NVIDIA A100 GPU of a Colab runtime has 40 GB. Storing numbers in a lower precision reduces the memory, since single precision takes half the bytes of double precision and the 16-bit formats a quarter. A naive switch to low precision is unsafe, however, because rounding can destroy the positive definiteness of an ill-conditioned covariance matrix. This covariance matrix is ill-conditioned, since the forces are smooth and strongly correlated along the gait cycle and across strides while the nugget effect is about a thousandth of the variance.

The notebook therefore implements the adaptive block mixed-precision Cholesky factorization of Salvaña (2026). The covariance matrix is never stored. Its blocks are formed on demand in double precision, the diagonal blocks of the Cholesky factor are kept in double precision, and every off-diagonal block of the factor is stored in the lowest precision whose measured rounding error fits within a budget that depends on a lower bound for the smallest eigenvalue of the covariance matrix. For this model the bound is the smallest nugget effect, which is available at no cost. The budget guarantees that the factorization runs to completion and that the covariance matrix implied by the stored factor differs from the exact one by less than a chosen fraction of that smallest eigenvalue. All arithmetic is done in double precision, and the factor blocks that do not fit in GPU memory are kept in host memory and copied back when they are needed.

The notebook follows the order of this argument. It loads the data, states the model and fits its two Kronecker special cases, which serve as likelihood baselines, as starting values and as an exact check. It then implements the factorization and verifies it in three ways, against a dense double-precision Cholesky factorization on a subset of 96 strides, against the error bound of the method, and against the exact Kronecker likelihood on all 145,440 points. It ends with the maximum likelihood estimation of the trivariate model. Prediction for the held-out participants is the subject of the next notebook.

This notebook requires an NVIDIA A100 GPU. In Colab, choose Runtime, then Change runtime type, then A100 GPU with High-RAM, before running it.

## Setup

Load the libraries and confirm that the runtime is connected to an A100 GPU. The 40 GB card holds part of the Cholesky factor and the host memory holds the rest, so the High-RAM runtime is needed as well.

In [ ]:
import json, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import torch
from scipy.optimize import minimize

assert torch.cuda.is_available(), "No GPU is attached. Choose Runtime > Change runtime type > A100 GPU with High-RAM."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
RAM_GB = psutil.virtual_memory().total / 1e9
print(f"GPU {GPU_NAME}, {GPU_GB:.1f} GB    host memory {RAM_GB:.1f} GB    PyTorch {torch.__version__}")
assert "A100" in GPU_NAME, f"This notebook is written for an NVIDIA A100 GPU, but the runtime has a {GPU_NAME}."
assert GPU_GB + RAM_GB > 75, "The Cholesky factor needs 43 GB between the GPU and the host. Choose the High-RAM runtime."

## Data

The data are those of the first notebook. One row is one stride of the right leg, 480 training strides from twelve participants and 140 test strides from three held-out participants. The inputs are three ground reaction force curves, five joint angle curves, the belt speed, the body mass and the height. The outputs are the hip, knee and ankle joint contact force curves computed by OpenSim, in body weights (BW). Every curve is time normalized to 101 points of the gait cycle, from one heel strike to the next. Only the training strides are used in this notebook.

In [ ]:
!gdown --folder "https://drive.google.com/drive/folders/1CI5OAxzB9qYI9vIxW1f0cGE4SN1hhyBj" -q

In [ ]:
x_train = pd.read_csv("data/x-train.csv")
y_train = pd.read_csv("data/y-train.csv")
x_train.shape, y_train.shape

Each stride becomes the vector $\mathbf{x}_i$ of its eight input curves and three scalars, standardized on the training strides exactly as in the first notebook, so that all 811 coordinates are on a comparable scale.

In [ ]:
GRID = np.arange(101)
JOINTS = ["hip", "knee", "ankle"]
INPUT_CURVES = ["grf_vertical_BW", "grf_anteroposterior_BW", "grf_mediolateral_BW",
                "hip_flexion_deg", "hip_adduction_deg", "hip_rotation_deg", "knee_flexion_deg", "ankle_angle_deg"]

def curve(df, name):
    return df[[f"{name}_t{t:03d}" for t in GRID]].to_numpy()

def build_features(x, x_ref):
    cols = []
    for name in INPUT_CURVES:
        C, C_ref = curve(x, name), curve(x_ref, name)
        cols.extend(((C - C_ref.mean(axis=0)) / C_ref.std()).T)
    for name in ["belt_speed_mps", "mass_kg", "height_m"]:
        cols.append((x[name].to_numpy() - x_ref[name].mean()) / x_ref[name].std())
    return np.array(cols).T

F_train = build_features(x_train, x_train)
F_train.shape

The Gaussian process models the residual of each joint, the contact force curve minus the mean curve $m_j(t)$ of the training strides. The residuals of joint $j$ are divided by the standard deviation $s_j$ of its training curves, as in the first notebook, which puts the three joints on one scale. All likelihoods in this notebook are likelihoods of these standardized residuals, so they are comparable with each other and with the univariate fits of the first notebook.

In [ ]:
Y_train = np.stack([curve(y_train, f"{joint}_jcf_BW") for joint in JOINTS])      # (3 joints, 480 strides, 101 gait points)
m_t = Y_train.mean(axis=1)                                                        # mean curve of each joint
scale = Y_train.std(axis=(1, 2))                                                  # s_j, in BW
Z_train = (Y_train - m_t[:, None, :]) / scale[:, None, None]                      # standardized residuals
Z_train.shape, scale.round(3)

The correlations between the standardized residuals of the three joints, taken over all strides and gait points, show the dependence that three separate univariate emulators ignore.

In [ ]:
pd.DataFrame(np.corrcoef(Z_train.reshape(3, -1)), index=JOINTS, columns=JOINTS).round(3)

## Trivariate separable space-time Matérn model

Let $i = 1, \dots, n$ index the training strides, with $n = 480$, let $t \in \{0, 1, \dots, 100\}$ denote the percent of the gait cycle, and let $j \in \{1, 2, 3\}$ index the hip, the knee and the ankle. The input of stride $i$ is the vector $\mathbf{x}_i \in \mathbb{R}^{811}$, and $y_{ij}(t)$ is the contact force of joint $j$ at gait point $t$ of stride $i$. Each joint follows the same decomposition as in the univariate emulator,
$$
y_{ij}(t) = m_j(t) + f_j(\mathbf{x}_i, t) + \varepsilon_{ij}(t), \qquad \varepsilon_{ij}(t) \sim \mathcal{N}(0, \sigma_{\varepsilon, j}^2),
$$
where $m_j(t)$ is the mean curve of joint $j$ and $\varepsilon_{ij}(t)$ is independent noise whose variance $\sigma_{\varepsilon, j}^2$ is the nugget effect of joint $j$. The difference is that $(f_1, f_2, f_3)$ is now a trivariate Gaussian process. It is specified by a matrix-valued covariance function, which gives the covariance between any two joints at any two points,
$$
\operatorname{Cov}\!\big(f_j(\mathbf{x}, t), f_k(\mathbf{x}', t')\big) = C_{jk}\big((\mathbf{x}, t), (\mathbf{x}', t')\big).
$$

Write $h$ for the root mean squared difference between the standardized inputs of two strides, $h^2 = \frac{1}{p} \sum_{q=1}^{p} (x_q - x'_q)^2$ with $p = 811$, and $u = |t - t'|$ for the lag between two gait points in percent of the gait cycle. The covariance function is the space-time multivariate Matérn with smoothness $5/2$,
$$
C_{jk}\big((\mathbf{x}, t), (\mathbf{x}', t')\big) = \beta_{jk}\, \sigma_j \sigma_k \;\, \gamma_{jk}\, \mathcal{M}\!\Big(\frac{h}{\ell_{jk}}\Big) \;\, \gamma_{t, jk}\, \mathcal{M}\!\Big(\frac{u}{\ell_{t, jk}}\Big), \qquad \mathcal{M}(d) = \Big(1 + \sqrt{5}\, d + \tfrac{5}{3} d^2\Big) e^{-\sqrt{5}\, d}.
$$
Joint $j$ has its own variance $\sigma_j^2$, input range $\ell_j$ and temporal range $\ell_{t, j}$. The cross-correlation parameters $\beta_{jk}$ form a correlation matrix, with $\beta_{jj} = 1$. For a pair of joints the range and the factor $\gamma$ are determined by the ranges of the two joints,
$$
\frac{1}{\ell_{jk}^2} = \frac{1}{2} \Big( \frac{1}{\ell_j^2} + \frac{1}{\ell_k^2} \Big), \qquad \gamma_{jk} = \Big( \frac{2\, \ell_j \ell_k}{\ell_j^2 + \ell_k^2} \Big)^{5/2},
$$
and $\ell_{t, jk}$ and $\gamma_{t, jk}$ are defined in the same way from the temporal ranges. With a common smoothness this construction is a valid model for every correlation matrix $(\beta_{jk})$ and in any input dimension, which matters here because the input has 811 coordinates (Gneiting, Kleiber and Schlather, 2010, and Apanasovich, Genton and Sun, 2012).

For a single joint $\ell_{jj} = \ell_j$ and $\gamma_{jj} = 1$, so that $C_{jj} = \sigma_j^2\, \mathcal{M}(h / \ell_j)\, \mathcal{M}(u / \ell_{t, j})$. This is the separable space-time covariance function of the univariate emulator, and the two ranges keep their reading. The input range $\ell_j$ is the root mean squared difference in standardized inputs at which the correlation between two strides has fallen to $0.52$, and the temporal range $\ell_{t, j}$ is the same quantity for two gait points, in percent of the gait cycle. Every entry $C_{jk}$ is likewise the product of a function of the inputs and a function of gait time, so the model is separable in space and time. It is not separable across joints, because the ranges change with the pair of joints. The factor $\gamma_{jk} \le 1$ equals one only when the two joints have the same range, and the cross-correlation between the forces of joints $j$ and $k$ at the same stride and gait point is
$$
\rho_{jk} = \beta_{jk}\, \gamma_{jk}\, \gamma_{t, jk}.
$$

Two special cases are important. When $\beta_{jk} = 0$ for every pair of joints the model is three independent univariate emulators, and its likelihood is the sum of the three univariate likelihoods. When the three joints share one input range $\ell$ and one temporal range $\ell_t$ every $\gamma$ equals one, and with the points ordered by joint, then stride, then gait point, the covariance matrix of the stacked residuals is
$$
\boldsymbol{\Sigma} = \mathbf{B} \otimes \boldsymbol{\Sigma}_x \otimes \boldsymbol{\Sigma}_t + \mathbf{D} \otimes \mathbf{I}, \qquad \mathbf{B} = \big(\beta_{jk}\, \sigma_j \sigma_k\big), \quad \mathbf{D} = \operatorname{diag}\big(\sigma_{\varepsilon, 1}^2, \sigma_{\varepsilon, 2}^2, \sigma_{\varepsilon, 3}^2\big),
$$
where $\boldsymbol{\Sigma}_x$ is the $n \times n$ matrix of $\mathcal{M}(h / \ell)$ between training strides and $\boldsymbol{\Sigma}_t$ the $101 \times 101$ matrix of $\mathcal{M}(u / \ell_t)$ between gait points. This Kronecker model is known in the multivariate literature as the intrinsic coregionalization model. Both special cases can be fitted in seconds from the eigendecompositions of small matrices. The general model cannot, because joint-specific ranges leave no Kronecker structure to exploit.

The parameters are $\boldsymbol{\theta} = \{\ell_j, \ell_{t, j}, \sigma_j^2, \sigma_{\varepsilon, j}^2, \beta_{jk}\}$, fifteen numbers in all. They are estimated by maximizing the log likelihood of the stacked residuals $\mathbf{y}$, a vector of length $N = 3 \times 480 \times 101 = 145{,}440$,
$$
\log p(\mathbf{y} \mid \boldsymbol{\theta}) = -\tfrac{1}{2}\, \mathbf{y}^\top \boldsymbol{\Sigma}^{-1} \mathbf{y} - \tfrac{1}{2} \log \det \boldsymbol{\Sigma} - \tfrac{N}{2} \log 2\pi .
$$
One parameter can be removed analytically. Write $\boldsymbol{\Sigma} = \sigma_1^2\, \bar{\boldsymbol{\Sigma}}$, where $\bar{\boldsymbol{\Sigma}}$ depends on the ranges, the cross-correlation parameters, the variance ratios $v_j = \sigma_j^2 / \sigma_1^2$ and the relative nugget effects $\eta_j = \sigma_{\varepsilon, j}^2 / \sigma_1^2$. For fixed values of these fourteen parameters the likelihood is maximized by $\hat{\sigma}_1^2 = \mathbf{y}^\top \bar{\boldsymbol{\Sigma}}^{-1} \mathbf{y} / N$, and the negative log likelihood becomes
$$
-\log p = \tfrac{1}{2} \Big( N \log \hat{\sigma}_1^2 + \log \det \bar{\boldsymbol{\Sigma}} + N + N \log 2\pi \Big).
$$
Each evaluation therefore needs the log determinant of $\bar{\boldsymbol{\Sigma}}$ and one linear solve, and both come from its Cholesky factor.

## Special cases with Kronecker structure

The class below evaluates the exact likelihood of the Kronecker model $\mathbf{B} \otimes \boldsymbol{\Sigma}_x \otimes \boldsymbol{\Sigma}_t + \mathbf{D} \otimes \mathbf{I}$ for any number of joints from the eigendecompositions of $\mathbf{B}$, $\boldsymbol{\Sigma}_x$ and $\boldsymbol{\Sigma}_t$. With one joint it is the univariate emulator of the first notebook, and with three joints it is the equal-range special case. The correlation matrix $(\beta_{jk})$ is parameterized by three unconstrained numbers through the angles of its Cholesky factor, so that every value of the parameters gives a valid correlation matrix.

In [ ]:
def matern_np(d):
    s = np.sqrt(5.0) * d
    return (1.0 + s + s**2 / 3.0) * np.exp(-s)

def correlation_matrix(raw, J):
    # correlation matrix from J (J - 1) / 2 unconstrained numbers: rows of its Cholesky factor in spherical coordinates
    angles = (np.pi / 2) * np.tanh(np.asarray(raw, dtype=float))
    L = np.zeros((J, J)); L[0, 0] = 1.0; q = 0
    for i in range(1, J):
        remaining = 1.0
        for j in range(i):
            L[i, j] = remaining * np.sin(angles[q]); remaining *= np.cos(angles[q]); q += 1
        L[i, i] = remaining
    return L @ L.T

def rms_distance(F):
    G = F @ F.T
    sq = np.diag(G)[:, None] + np.diag(G)[None, :] - 2 * G
    H = np.sqrt(np.maximum(sq, 0.0) / F.shape[1])        # root mean squared difference between the inputs of two strides
    np.fill_diagonal(H, 0.0)
    return H

class KroneckerGP:
    # theta = log input range, log temporal range, correlation angles, log variances (J), log nugget effects (J)
    def __init__(self, F, Z, t):
        self.J, self.n, self.T = Z.shape
        self.Z, self.t, self.H = Z, t, rms_distance(F)
        self.n_corr = self.J * (self.J - 1) // 2

    def unpack(self, theta):
        J, c = self.J, self.n_corr
        return np.exp(theta[0]), np.exp(theta[1]), correlation_matrix(theta[2:2 + c], J), np.exp(theta[2 + c:2 + c + J]), np.exp(theta[2 + c + J:])

    def spectrum(self, theta):
        l, lt, beta, var, nugget = self.unpack(theta)
        sd = np.sqrt(nugget)
        B = beta * np.sqrt(np.outer(var, var)) / np.outer(sd, sd)                 # B scaled by the nugget standard deviations
        b, W = np.linalg.eigh(B)
        lam, U = np.linalg.eigh(matern_np(self.H / l) + 1e-10 * np.eye(self.n))
        mu, V = np.linalg.eigh(matern_np(np.abs(self.t[:, None] - self.t[None, :]) / lt) + 1e-10 * np.eye(self.T))
        return sd, (b, W), (lam, U), (mu, V)

    def nll(self, theta):
        sd, (b, W), (lam, U), (mu, V) = self.spectrum(theta)
        D = b[:, None, None] * lam[None, :, None] * mu[None, None, :] + 1.0       # eigenvalues of the scaled covariance matrix
        if np.any(D <= 0):
            return 1e25
        R = np.stack([U.T @ (self.Z[j] / sd[j]) @ V for j in range(self.J)])
        R = np.tensordot(W.T, R, axes=(1, 0))
        N = self.J * self.n * self.T
        return 0.5 * np.sum(R**2 / D) + 0.5 * np.sum(np.log(D)) + self.n * self.T * np.sum(np.log(sd)) + 0.5 * N * np.log(2 * np.pi)

def fit_kronecker(gp, theta0, maxiter=400):
    return minimize(gp.nll, theta0, method="L-BFGS-B", options={"maxiter": maxiter, "eps": 1e-3})

### Three independent univariate emulators

The first special case is fitted one joint at a time. It reproduces the univariate fits of the first notebook, and the sum of the three negative log likelihoods is the baseline without cross-correlations.

In [ ]:
t_grid = GRID.astype(float)
univariate, rows = {}, []
for j, joint in enumerate(JOINTS):
    gp = KroneckerGP(F_train, Z_train[j:j + 1], t_grid)
    res = fit_kronecker(gp, np.log([2.0, 10.0, 1.0, 0.01]))
    l, lt, _, var, nugget = gp.unpack(res.x)
    univariate[joint] = dict(l=l, lt=lt, var=var[0], nugget=nugget[0], nll=res.fun)
    rows.append(pd.Series({"input range [standardized units]": l, "temporal range [% gait cycle]": lt, "variance [BW²]": var[0] * scale[j]**2,
                           "nugget effect [BW²]": nugget[0] * scale[j]**2, "negative log likelihood": res.fun}, name=joint))
nll_independent = sum(u["nll"] for u in univariate.values())
table = pd.DataFrame(rows).T
table.columns.name = "joint"
shown = table.apply(lambda col: col.map("{:.3f}".format))
shown.loc["nugget effect [BW²]"] = table.loc["nugget effect [BW²]"].map("{:.2e}".format)
print(f"negative log likelihood of the three independent univariate emulators: {nll_independent:,.1f}")
shown

### Equal ranges for the three joints

The second special case models the three joints jointly with cross-correlations, but with one input range and one temporal range for all of them. It starts from the average of the univariate ranges.

In [ ]:
kron = KroneckerGP(F_train, Z_train, t_grid)
theta0 = np.concatenate([[np.mean([np.log(u["l"]) for u in univariate.values()]), np.mean([np.log(u["lt"]) for u in univariate.values()])],
                         0.3 * np.ones(3), [np.log(u["var"]) for u in univariate.values()], [np.log(u["nugget"]) for u in univariate.values()]])
t0 = time.time()
res = fit_kronecker(kron, theta0)
theta_equal, nll_equal = res.x, res.fun
l_eq, lt_eq, beta_eq, var_eq, nugget_eq = kron.unpack(theta_equal)
print(f"equal ranges: negative log likelihood {nll_equal:,.1f}    input range {l_eq:.3f}    temporal range {lt_eq:.2f} % gait cycle    [{time.time() - t0:.0f} s]")
print(f"gain over the independent emulators: {nll_independent - nll_equal:,.1f} log likelihood units")
pd.DataFrame(beta_eq, index=JOINTS, columns=JOINTS).round(3)

## The dense covariance matrix

With joint-specific ranges the Kronecker structure is lost, and $\bar{\boldsymbol{\Sigma}}$ has to be factored as a dense $N \times N$ matrix. The cell below compares the memory this takes with the memory of the GPU. A matrix entry takes 8 bytes in double precision (`fp64`), 4 bytes in single precision (`fp32`), 2 bytes in `bfloat16` (`bf16`) and 1 byte in the 8-bit format `fp8`.

In [ ]:
N = Z_train.size
memory = pd.Series({"fp64 covariance matrix": N * N * 8 / 1e9, "fp64 Cholesky factor (lower triangle)": N * (N + 1) / 2 * 8 / 1e9,
                    "fp32 Cholesky factor (lower triangle)": N * (N + 1) / 2 * 4 / 1e9, "memory of this GPU": GPU_GB}, name="GB")
print(f"N = {N:,}")
memory.round(1).to_frame()

A lower precision saves memory at the cost of rounding. A floating-point format rounds a real number $x$ to a representable number $\mathrm{fl}(x) = x (1 + \delta)$ with $|\delta| \le u$, where $u$ is the unit roundoff of the format, provided that $x$ is neither too large nor too small for the format. Rounding the entries of a covariance matrix perturbs its eigenvalues. A sufficient condition for the rounded matrix to remain positive definite is that the condition number, the ratio $\kappa_2 = \lambda_{\max} / \lambda_{\min}$ of the largest to the smallest eigenvalue, is below $1 / u$.

The condition number of the full matrix is available exactly in the equal-range special case, from the eigenvalues of the three small matrices. Its smallest eigenvalue is the smallest relative nugget effect to within a negligible amount, because $\mathbf{B} \otimes \boldsymbol{\Sigma}_x \otimes \boldsymbol{\Sigma}_t$ has eigenvalues that are numerically zero. The table compares the condition number with $1 / u$ for each format.

In [ ]:
B_bar = beta_eq * np.sqrt(np.outer(var_eq, var_eq)) / var_eq[0]                   # B with the hip variance factored out
eta_eq = nugget_eq / var_eq[0]                                                    # relative nugget effects
lam_x = np.linalg.eigvalsh(matern_np(kron.H / l_eq))
lam_t = np.linalg.eigvalsh(matern_np(np.abs(t_grid[:, None] - t_grid[None, :]) / lt_eq))
lambda_max = np.linalg.eigvalsh(B_bar).max() * lam_x.max() * lam_t.max() + eta_eq.max()
condition_number = lambda_max / eta_eq.min()
print(f"largest eigenvalue {lambda_max:,.0f}    smallest relative nugget effect {eta_eq.min():.2e}    condition number {condition_number:.1e}")

UNIT_ROUNDOFF = {"fp8": 2.0**-4, "bf16": 2.0**-8, "fp32": 2.0**-24, "fp64": 2.0**-53}
BYTES = {"fp8": 1, "bf16": 2, "fp32": 4, "fp64": 8}
LARGEST = {"fp8": 448.0, "bf16": 3.39e38, "fp32": 3.40e38, "fp64": 1.80e308}
formats = pd.DataFrame({"bytes per entry": BYTES, "unit roundoff u": UNIT_ROUNDOFF, "1 / u": {k: 1 / v for k, v in UNIT_ROUNDOFF.items()},
                        "largest finite number": LARGEST})
formats["condition number below 1 / u"] = formats["1 / u"] > condition_number
formats.style.format({"unit roundoff u": "{:.1e}", "1 / u": "{:.1e}", "largest finite number": "{:.1e}"})

The condition number is in the millions. It exceeds $1 / u$ of `fp8` and `bf16` by several orders of magnitude, so neither format can be used blindly, and it is within a factor of five of the limit of `fp32`. The lower formats are still usable for the parts of the factor that are small enough, and deciding which parts those are is the purpose of the adaptive rule in the next section.

## Adaptive block mixed-precision Cholesky factorization

The factorization follows Algorithm 1 of Salvaña (2026). Partition $\bar{\boldsymbol{\Sigma}}$ into an $n_B \times n_B$ grid of blocks of $n_b$ consecutive points. Let $m$ be a known lower bound on the smallest eigenvalue, $0 < m \le \lambda_{\min}(\bar{\boldsymbol{\Sigma}})$. Here $\bar{\boldsymbol{\Sigma}}$ is a nonnegative definite matrix plus a diagonal matrix that holds the relative nugget effects, so $m = \min_j \eta_j$ is available at no cost. For an error-budget parameter $0 < \tau < 1$ the budget of each block is
$$
b = \frac{\tau\, m}{n_B}.
$$

Let $\widetilde{\mathbf{L}}$ denote the stored factor. At block column $k$ the diagonal block is computed and kept in `fp64`,
$$
\widetilde{\mathbf{L}}_{kk} = \operatorname{chol}\Big( \bar{\boldsymbol{\Sigma}}_{kk} - \sum_{j < k} \widetilde{\mathbf{L}}_{kj} \widetilde{\mathbf{L}}_{kj}^\top \Big),
$$
and for each $i > k$ the off-diagonal block is computed in `fp64` as
$$
\mathbf{A}_{ik} = \Big( \bar{\boldsymbol{\Sigma}}_{ik} - \sum_{j < k} \widetilde{\mathbf{L}}_{ij} \widetilde{\mathbf{L}}_{kj}^\top \Big) \widetilde{\mathbf{L}}_{kk}^{-\top}.
$$
The block $\mathbf{A}_{ik}$ is then converted to each candidate format $p$ in the order `fp8`, `bf16`, `fp32`, read back in `fp64`, and stored in the first format with finite entries that satisfies
$$
\big\| \mathrm{fl}_p(\mathbf{A}_{ik}) - \mathbf{A}_{ik} \big\|_F\, \big\| \widetilde{\mathbf{L}}_{kk} \big\|_2 \le b,
$$
where $\|\cdot\|_F$ is the Frobenius norm and $\|\cdot\|_2$ the spectral norm. If no candidate qualifies the block stays in `fp64`. The left-hand side bounds the change that storing the block in format $p$ causes in the covariance matrix implied by the stored factor, $\widetilde{\boldsymbol{\Sigma}} = \widetilde{\mathbf{L}} \widetilde{\mathbf{L}}^\top$. Because the test uses the measured error of the conversion, it remains valid when entries underflow in the candidate format.

Under this rule, and treating `fp64` arithmetic as exact, the Cholesky factorization of every diagonal block exists, so the factorization cannot break down, and the completed factor satisfies
$$
\big\| \widetilde{\boldsymbol{\Sigma}} - \bar{\boldsymbol{\Sigma}} \big\|_F \le \tau\, m \le \tau\, \lambda_{\min}(\bar{\boldsymbol{\Sigma}}),
$$
which implies $\lambda_{\min}(\widetilde{\boldsymbol{\Sigma}}) \ge (1 - \tau)\, \lambda_{\min}(\bar{\boldsymbol{\Sigma}}) > 0$. The implementation also accumulates the measured errors $e_{ik} = \| \mathrm{fl}_p(\mathbf{A}_{ik}) - \mathbf{A}_{ik} \|_F \| \widetilde{\mathbf{L}}_{kk} \|_2$ of the stored blocks. Since the diagonal blocks of $\widetilde{\boldsymbol{\Sigma}} - \bar{\boldsymbol{\Sigma}}$ are zero and each off-diagonal block has Frobenius norm at most $e_{ik}$, the quantity $\big( 2 \sum_{i > k} e_{ik}^2 \big)^{1/2}$ is a computed bound on $\| \widetilde{\boldsymbol{\Sigma}} - \bar{\boldsymbol{\Sigma}} \|_F$ that is obtained without ever forming either matrix.

Two features of the implementation serve the scale of the problem. Covariance blocks are formed when they are needed and discarded after use, so the 169 GB matrix never exists. When the stored factor exceeds the share of GPU memory set aside for it, further blocks are written to page-locked host memory and copied back to the GPU when the recurrence reads them. The block columns that are stored last are the ones read least often, so the transfers add little time. Offloading changes where a block is kept, not its precision or the arithmetic. The log determinant is accumulated from the diagonal blocks, $\log \det \widetilde{\boldsymbol{\Sigma}} = 2 \sum_k \sum_r \log (\widetilde{\mathbf{L}}_{kk})_{rr}$.

### Covariance blocks

The first class holds the model. It orders the points by joint, then stride, then gait point, and returns any block of $\bar{\boldsymbol{\Sigma}}$ without its nugget effect, computed on the GPU in `fp64`. Its parameter vector holds the logarithms of the three input ranges and the three temporal ranges, the three correlation angles, the logarithms of the variance ratios of the knee and the ankle to the hip, and the logarithms of the three relative nugget effects.

In [ ]:
SQRT5 = 5.0**0.5

def matern(d):
    s = SQRT5 * d
    return (1.0 + s + s * s / 3.0) * torch.exp(-s)

def pair_ranges(l):
    # inverse pairwise ranges 1 / l_jk and the factors gamma_jk from the ranges of the joints
    a2 = 1.0 / l**2
    inverse = np.sqrt((a2[:, None] + a2[None, :]) / 2)
    return inverse, (np.sqrt(a2[:, None] * a2[None, :]) / inverse**2) ** 2.5

class TrivariateMatern:
    def __init__(self, F, Z, t, device):
        self.J, self.n, self.T = Z.shape
        self.N, self.n_corr, self.device = Z.size, self.J * (self.J - 1) // 2, device
        Ft = torch.tensor(F, dtype=torch.float64, device=device)
        self.H = torch.cdist(Ft, Ft, compute_mode="donot_use_mm_for_euclid_dist") / F.shape[1]**0.5      # h between strides
        self.t = torch.tensor(t, dtype=torch.float64, device=device)
        self.z = torch.tensor(Z.reshape(-1), dtype=torch.float64, device=device)                          # stacked residuals
        index = torch.arange(self.N, device=device)
        self.joint, self.stride, self.time = index // (self.n * self.T), (index // self.T) % self.n, index % self.T

    def set_parameters(self, theta):
        J, c = self.J, self.n_corr
        l, lt = np.exp(theta[:J]), np.exp(theta[J:2 * J])
        beta = correlation_matrix(theta[2 * J:2 * J + c], J)
        v = np.concatenate([[1.0], np.exp(theta[2 * J + c:3 * J + c - 1])])        # variance ratios to the first joint
        eta = np.exp(theta[3 * J + c - 1:])                                        # relative nugget effects
        inverse_l, gamma = pair_ranges(l)
        inverse_lt, gamma_t = pair_ranges(lt)
        rho = beta * gamma * gamma_t                                               # cross-correlations at the same stride and gait point
        as_tensor = lambda a: torch.tensor(a, dtype=torch.float64, device=self.device)
        self.amplitude = as_tensor(rho * np.sqrt(np.outer(v, v)))
        self.inverse_l, self.inverse_lt = as_tensor(inverse_l), as_tensor(inverse_lt)
        self.eta_point = as_tensor(eta)[self.joint]                                # nugget effect of every point
        self.m = float(eta.min())                                                  # lower bound on the smallest eigenvalue
        self.parameters = dict(l=l, lt=lt, beta=beta, v=v, eta=eta, rho=rho)

    def block(self, rows, cols):
        pair = (self.joint[rows][:, None], self.joint[cols][None, :])
        h = self.H[self.stride[rows]][:, self.stride[cols]]
        u = (self.t[self.time[rows]][:, None] - self.t[self.time[cols]][None, :]).abs()
        return self.amplitude[pair] * matern(h * self.inverse_l[pair]) * matern(u * self.inverse_lt[pair])

### Factorization

The second class is the factorization. `choose` is the precision rule, `store` and `load` keep a block on the GPU or in host memory and return it in `fp64`, `factor` is the block recurrence, and `solve` applies the inverse of $\widetilde{\mathbf{L}} \widetilde{\mathbf{L}}^\top$ by block forward and back substitution. The argument `storage="adaptive"` selects the rule above. Passing the name of one format instead stores every off-diagonal block in that format regardless of the budget, which is the naive use of low precision that the verification section compares against.

In [ ]:
FORMATS = ["fp8", "bf16", "fp32", "fp64"]
DTYPE = {"fp8": torch.float8_e4m3fn, "bf16": torch.bfloat16, "fp32": torch.float32, "fp64": torch.float64}
CANDIDATES = ["fp8", "bf16", "fp32"]            # tested in this order, fp64 is the fallback

class Breakdown(RuntimeError):
    pass

class BlockMixedCholesky:
    def __init__(self, N, block_size=1024, tau=0.99, storage="adaptive", gpu_factor_gb=None):
        self.blocks = [slice(a, min(a + block_size, N)) for a in range(0, N, block_size)]
        self.nB, self.tau, self.storage, self.gpu_factor_gb = len(self.blocks), tau, storage, gpu_factor_gb
        self.L, self.host = {}, {}              # stored factor blocks, and reusable page-locked host buffers

    def choose(self, A, budget):
        # lowest precision whose measured storage error fits the budget, with fp64 as the fallback
        if self.storage == "fp64":
            return "fp64", 0.0
        if self.storage != "adaptive":
            return self.storage, float(torch.linalg.norm(A.to(DTYPE[self.storage]).to(torch.float64) - A))
        largest = float(A.abs().max())
        for name in CANDIDATES:
            if largest > LARGEST[name]:
                continue
            error = float(torch.linalg.norm(A.to(DTYPE[name]).to(torch.float64) - A))
            if error <= budget:
                return name, error
        return "fp64", 0.0

    def store(self, i, k, block, name):
        self.bytes[name] += block.nbytes
        if block.device.type != "cuda" or self.gpu_factor_gb is None or self.gpu_bytes + block.nbytes <= self.gpu_factor_gb * 1e9:
            self.L[i, k] = block
            self.gpu_bytes += block.nbytes if block.device.type == "cuda" else 0
            return
        buffer = self.host.get((i, k))
        if buffer is None or buffer.shape != block.shape or buffer.dtype != block.dtype:
            buffer = torch.empty(block.shape, dtype=block.dtype, pin_memory=True)
            self.host[i, k] = buffer
        buffer.copy_(block)
        self.L[i, k] = buffer
        self.host_bytes += block.nbytes

    def load(self, i, k):
        return self.L[i, k].to(self.device, non_blocking=True).to(torch.float64)

    def factor(self, cov):
        nB, blocks, self.device = self.nB, self.blocks, cov.device
        b = self.tau * cov.m / nB                                                  # budget of each block
        self.L, self.format = {}, np.full((nB, nB), -1)
        self.bytes = dict.fromkeys(FORMATS, 0)
        self.gpu_bytes = self.host_bytes = 0
        error2, logdet = 0.0, 0.0
        for k in range(nB):
            S = cov.block(blocks[k], blocks[k])
            S.diagonal().add_(cov.eta_point[blocks[k]])
            row = [self.load(k, j) for j in range(k)]                              # block row k of the factor, in fp64
            for Lkj in row:
                S.addmm_(Lkj, Lkj.T, alpha=-1.0)
            Lkk, info = torch.linalg.cholesky_ex(S)
            del S
            if int(info) != 0 or not bool(torch.isfinite(Lkk).all()):
                raise Breakdown(f"diagonal block {k + 1} of {nB} is not positive definite")
            self.format[k, k] = FORMATS.index("fp64")
            self.store(k, k, Lkk, "fp64")
            logdet += 2.0 * float(torch.log(torch.diagonal(Lkk)).sum())
            norm_kk = float(torch.linalg.matrix_norm(Lkk, ord=2))
            for i in range(k + 1, nB):
                A = cov.block(blocks[i], blocks[k])
                for j in range(k):
                    A.addmm_(self.load(i, j), row[j].T, alpha=-1.0)
                A = torch.linalg.solve_triangular(Lkk.T, A, upper=True, left=False)
                name, error = self.choose(A, b / norm_kk)
                self.format[i, k] = FORMATS.index(name)
                error2 += (error * norm_kk) ** 2
                self.store(i, k, A.to(DTYPE[name]), name)
                del A
            del row
        self.logdet, self.budget = logdet, self.tau * cov.m
        self.perturbation_bound = (2.0 * error2) ** 0.5                            # computed bound on || L L^T - Sigma ||_F
        return logdet

    def solve(self, rhs):
        x = rhs.clone().reshape(rhs.shape[0], -1)
        for k, bk in enumerate(self.blocks):                                       # forward substitution
            s = x[bk]
            for j in range(k):
                s -= self.load(k, j) @ x[self.blocks[j]]
            x[bk] = torch.linalg.solve_triangular(self.L[k, k], s, upper=False)
        for k in range(self.nB - 1, -1, -1):                                       # back substitution
            bk = self.blocks[k]
            s = x[bk]
            for i in range(k + 1, self.nB):
                s -= self.load(i, k).T @ x[self.blocks[i]]
            x[bk] = torch.linalg.solve_triangular(self.L[k, k].T, s, upper=True)
        return x.reshape(rhs.shape)

    def mix(self):
        # percentage of the off-diagonal factor blocks stored in each format
        off = self.format[np.tril_indices(self.nB, -1)]
        return {name: 100.0 * np.mean(off == q) if off.size else 0.0 for q, name in enumerate(FORMATS)}

    def stored_gb(self):
        return sum(self.bytes.values()) / 1e9

### Likelihood

The negative log likelihood with the hip variance profiled out needs one factorization and one solve.

In [ ]:
def negative_log_likelihood(theta, cov, chol):
    cov.set_parameters(theta)
    logdet = chol.factor(cov)
    alpha = chol.solve(cov.z)
    sigma2 = float(cov.z @ alpha) / cov.N                                          # profiled variance of the first joint
    return 0.5 * (cov.N * np.log(sigma2) + logdet + cov.N + cov.N * np.log(2 * np.pi)), sigma2

def describe(chol):
    return "/".join(f"{chol.mix()[name]:.0f}" for name in FORMATS)

def peak_gpu_gb():
    return torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else float("nan")

## Verification

### Against dense double precision and the error bound

The first check uses a subset that a dense `fp64` Cholesky factorization can handle, every fifth training stride with all 101 gait points and all three joints, $N = 3 \times 96 \times 101 = 29{,}088$, which is close to the largest problem for which a dense `fp64` matrix, its factor and the comparison fit on a 40 GB GPU. The parameters are set to a point where the model is not a Kronecker product. The input ranges and temporal ranges are those of the three univariate fits, and the cross-correlation parameters, variances and nugget effects are those of the equal-range fit.

For each way of storing the factor the table reports the difference between its negative log likelihood and the dense `fp64` value, and two measures of the perturbation $\| \widetilde{\boldsymbol{\Sigma}} - \bar{\boldsymbol{\Sigma}} \|_F$ in units of the budget $\tau m$. The first is exact, obtained by multiplying the stored factor back together and subtracting the covariance matrix. The second is the bound computed during the factorization. The guarantee of the method is that both are at most one for the adaptive rule. The remaining rows store every off-diagonal block in one format, which is what a naive use of low precision would do.

In [ ]:
def trivariate_theta(l, lt, angles, var, nugget):
    return np.concatenate([np.log(l), np.log(lt), angles, np.log(var[1:] / var[0]), np.log(nugget / var[0])])

theta_check = trivariate_theta(np.array([univariate[j]["l"] for j in JOINTS]), np.array([univariate[j]["lt"] for j in JOINTS]),
                               theta_equal[2:5], var_eq, nugget_eq)

subset = np.arange(0, F_train.shape[0], 5)
cov_small = TrivariateMatern(F_train[subset], Z_train[:, subset], t_grid, DEVICE)
cov_small.set_parameters(theta_check)
TAU = 0.99

def assemble(cov, blocks):
    # the dense covariance matrix, assembled from the same blocks that the factorization forms
    Sigma = torch.empty((cov.N, cov.N), dtype=torch.float64, device=cov.device)
    for a in blocks:
        for c in blocks:
            Sigma[a, c] = cov.block(a, c)
    Sigma.diagonal().add_(cov.eta_point)
    return Sigma

def perturbation(chol, Sigma):
    # exact || L L^T - Sigma ||_F, from the covariance matrix implied by the stored factor
    L = torch.zeros_like(Sigma)
    for (i, k) in chol.L:
        L[chol.blocks[i], chol.blocks[k]] = chol.load(i, k)
    implied = L @ L.T
    del L
    return float(torch.linalg.norm(implied.sub_(Sigma)))

reference = BlockMixedCholesky(cov_small.N, block_size=512, tau=TAU)
Sigma = assemble(cov_small, reference.blocks)
L_dense = torch.linalg.cholesky(Sigma)
alpha = torch.cholesky_solve(cov_small.z[:, None], L_dense)[:, 0]
sigma2 = float(cov_small.z @ alpha) / cov_small.N
nll_dense = 0.5 * (cov_small.N * np.log(sigma2) + 2.0 * float(torch.log(torch.diagonal(L_dense)).sum()) + cov_small.N + cov_small.N * np.log(2 * np.pi))
del L_dense, alpha
print(f"N = {cov_small.N:,}    dense fp64 negative log likelihood {nll_dense:,.4f}    budget tau m = {TAU * cov_small.m:.2e}")

rows, maps = [], {}
for storage in ["adaptive", "fp64", "fp32", "bf16", "fp8"]:
    chol = BlockMixedCholesky(cov_small.N, block_size=512, tau=TAU, storage=storage)
    label = "adaptive rule" if storage == "adaptive" else f"all {storage}"
    try:
        nll, _ = negative_log_likelihood(theta_check, cov_small, chol)
        exact = perturbation(chol, Sigma)
        rows.append(pd.Series({"factorization": "completed", "likelihood error": nll - nll_dense, "perturbation / budget": exact / chol.budget,
                               "computed bound / budget": chol.perturbation_bound / chol.budget, "stored factor [GB]": chol.stored_gb(),
                               "fp8/bf16/fp32/fp64 [%]": describe(chol)}, name=label))
        maps[label] = chol.format.copy()
    except Breakdown as failure:
        rows.append(pd.Series({"factorization": f"breakdown, {failure}"}, name=label))
    chol.L = {}
del Sigma
check = pd.DataFrame(rows)
check.style.format({"likelihood error": "{:+.2e}", "perturbation / budget": "{:.2e}", "computed bound / budget": "{:.2e}", "stored factor [GB]": "{:.2f}"}, na_rep="")

The adaptive rule reproduces the dense `fp64` likelihood to within the rounding error of the likelihood itself, and both measures of the perturbation are below the budget, as the guarantee requires. At these parameters the rule stores every off-diagonal block in `fp32`, which halves the stored factor. The strides are strongly correlated, so no block is small enough for a 16-bit or an 8-bit format.

The rows with a single format show why the rule is needed. Storing every block in `bf16` or in `fp8` ignores the budget. The factorization then either breaks down, because a diagonal block is no longer positive definite, or it completes with an implied covariance matrix far outside the budget and a likelihood error larger than the differences that the optimizer has to resolve between neighboring parameter values.

### The precision follows the dependence

The rule assigns a precision to each block from the size of the block and of the budget, so the mix of formats depends on how strongly the points of two blocks are correlated. The figure shows the format of every factor block at the parameters of the check above and again after the three input ranges are divided by ten, which makes most pairs of strides nearly independent. The likelihood error against dense `fp64` is given in each title.

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

FORMAT_COLORS = ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]          # one hue, light to dark with increasing precision

def precision_map(ax, fmt, title):
    ax.imshow(np.ma.masked_less(fmt, 0), cmap=ListedColormap(FORMAT_COLORS), vmin=-0.5, vmax=3.5, interpolation="nearest")
    ax.set_title(title, loc="left", fontsize=10)
    ax.set_xlabel("block column"); ax.set_ylabel("block row")

theta_weak = theta_check.copy()
theta_weak[:3] -= np.log(10.0)
cov_small.set_parameters(theta_weak)
chol = BlockMixedCholesky(cov_small.N, block_size=512, tau=TAU)
Sigma = assemble(cov_small, chol.blocks)
L_dense = torch.linalg.cholesky(Sigma)
alpha = torch.cholesky_solve(cov_small.z[:, None], L_dense)[:, 0]
sigma2 = float(cov_small.z @ alpha) / cov_small.N
nll_dense_weak = 0.5 * (cov_small.N * np.log(sigma2) + 2.0 * float(torch.log(torch.diagonal(L_dense)).sum()) + cov_small.N + cov_small.N * np.log(2 * np.pi))
del Sigma, L_dense, alpha
nll_weak, _ = negative_log_likelihood(theta_weak, cov_small, chol)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
precision_map(axes[0], maps["adaptive rule"], f"ranges of the univariate fits, likelihood error {check.loc['adaptive rule', 'likelihood error']:+.1e}")
precision_map(axes[1], chol.format, f"input ranges divided by ten, likelihood error {nll_weak - nll_dense_weak:+.1e}")
axes[1].legend(handles=[Patch(color=c, label=name) for c, name in zip(FORMAT_COLORS, FORMATS)], frameon=False, loc="upper right", title="stored format")
fig.tight_layout()
print(f"input ranges divided by ten: fp8/bf16/fp32/fp64 {describe(chol)} % of the off-diagonal blocks, stored factor {chol.stored_gb():.2f} GB, "
      f"computed bound / budget {chol.perturbation_bound / chol.budget:.2e}")
chol.L = {}
del cov_small

With short input ranges most pairs of strides are nearly independent, and the rule stores most of the off-diagonal blocks in `fp8` without any loss in the accuracy of the likelihood. The blocks that remain in `fp32` lie in bands. The band next to the diagonal holds neighboring blocks, which share a stride, and the two off-diagonal bands pair the same strides in two different joints, whose forces stay correlated whatever the input range. The precision of a block is therefore decided by the dependence between its points, not by its position in the matrix.

### Against the exact Kronecker likelihood on all 145,440 points

A dense `fp64` factorization of the full problem is out of reach, but the model contains a case whose likelihood is known exactly at full size. At the equal-range estimate the trivariate model is the Kronecker model, so the likelihood computed by the mixed-precision factorization of the dense $145{,}440 \times 145{,}440$ matrix must agree with the value computed from the three small eigendecompositions. The factorization does not use the Kronecker structure in any way. The cell reports the two values, the time of one evaluation, the peak GPU memory, the formats chosen by the rule and the computed bound on the perturbation.

On a 40 GB card, 60 % of the GPU memory is set aside for the stored factor and the remainder of the factor goes to host memory. The blocks have $n_b = 1024$ points, which gives $n_B = 143$.

In [ ]:
cov = TrivariateMatern(F_train, Z_train, t_grid, DEVICE)
chol = BlockMixedCholesky(cov.N, block_size=1024, tau=TAU, gpu_factor_gb=0.6 * GPU_GB)
theta_start = trivariate_theta(np.full(3, l_eq), np.full(3, lt_eq), theta_equal[2:5], var_eq, nugget_eq)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()                 # release the memory of the dense comparison
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()
nll_mixed, sigma2 = negative_log_likelihood(theta_start, cov, chol)
seconds = time.time() - t0
nll_exact = kron.nll(np.concatenate([theta_equal[:5], np.log(sigma2 * cov.parameters["v"]), np.log(sigma2 * cov.parameters["eta"])]))

fp64_gb = sum(chol.L[key].numel() for key in chol.L) * 8 / 1e9
pd.Series({"N": f"{cov.N:,}", "blocks": f"{chol.nB} x {chol.nB}",
           "negative log likelihood, mixed-precision dense factorization": f"{nll_mixed:,.4f}",
           "negative log likelihood, exact Kronecker": f"{nll_exact:,.4f}",
           "difference": f"{nll_mixed - nll_exact:+.4f}",
           "off-diagonal blocks in fp8/bf16/fp32/fp64 [%]": describe(chol),
           "computed perturbation bound / budget": f"{chol.perturbation_bound / chol.budget:.2e}",
           "stored factor [GB]": f"{chol.stored_gb():.1f}",
           "the same blocks in fp64 [GB]": f"{fp64_gb:.1f}",
           "factor on the GPU / in host memory [GB]": f"{chol.gpu_bytes / 1e9:.1f} / {chol.host_bytes / 1e9:.1f}",
           "peak GPU memory [GB]": f"{peak_gpu_gb():.1f}",
           "time of one likelihood evaluation [s]": f"{seconds:.0f}"}, name="full problem").to_frame()

## Maximum likelihood estimation

The fourteen parameters that remain after profiling are estimated with the L-BFGS-B optimizer, with gradients from forward differences of the likelihood. The optimizer starts at the equal-range estimate, where the trivariate likelihood equals the Kronecker likelihood, so every improvement from there is a gain of the trivariate model over the equal-range model. The parameters are bounded to a plausible region, input ranges between 0.05 and 5 standardized units, temporal ranges between 1 and 30 % of the gait cycle, and relative nugget effects between $10^{-5}$ and $0.1$.

Every evaluation of the likelihood is one mixed-precision factorization of the $145{,}440 \times 145{,}440$ matrix, and one iteration needs fifteen of them for the gradient, so the fit runs for hours. Two devices keep that time under control. A trial point that is farther than 0.7 on the logarithmic scale from the best point found so far is rejected without a factorization, which stops the line search from spending a full gradient far outside the region where the likelihood is informative. The best parameter vector is written to a checkpoint file after every improvement, and the cell resumes from that file when it exists, so the fit can be continued in a later session. Saving the checkpoint file to Google Drive keeps it between sessions.

In [ ]:
CHECKPOINT = "trivariate_matern_checkpoint.json"
MAX_ITERATIONS = 60
STEP_BOX = 0.7

bounds = [(np.log(0.05), np.log(5.0))] * 3 + [(np.log(1.0), np.log(30.0))] * 3 + [(-2.0, 2.0)] * 3 + [(-3.0, 3.0)] * 2 + [(np.log(1e-5), np.log(0.1))] * 3
best = {"nll": np.inf, "theta": None, "sigma2": None, "evaluations": 0, "skipped": 0}
if os.path.exists(CHECKPOINT):
    saved = json.load(open(CHECKPOINT))
    theta0 = np.array(saved["theta"])
    print(f"resuming from {CHECKPOINT}, negative log likelihood {saved['nll']:,.2f} after {saved['evaluations']} evaluations")
else:
    theta0 = theta_start.copy()
t_start = time.time()

def objective(theta):
    if best["theta"] is not None and np.max(np.abs(theta - best["theta"])) > STEP_BOX:
        best["skipped"] += 1                                                       # outside the step box, no factorization
        return best["nll"] + 1e6 * float(np.max(np.abs(theta - best["theta"])))
    t0 = time.time()
    try:
        nll, sigma2 = negative_log_likelihood(theta, cov, chol)
    except Breakdown as failure:
        print(f"  {failure}")
        return 1e12
    best["evaluations"] += 1
    p = cov.parameters
    improved = nll < best["nll"]
    if improved:
        best.update(nll=nll, theta=theta.copy(), sigma2=sigma2)
        json.dump({"theta": theta.tolist(), "nll": nll, "sigma2": sigma2, "evaluations": best["evaluations"]}, open(CHECKPOINT, "w"))
    print(f"evaluation {best['evaluations']:4d}  negative log likelihood {nll:13.2f}{' *' if improved else '  '}  input ranges {np.round(p['l'], 3).tolist()}  "
          f"temporal ranges {np.round(p['lt'], 2).tolist()}  cross-correlations {np.round(p['rho'][np.triu_indices(3, 1)], 3).tolist()}  "
          f"fp8/bf16/fp32/fp64 {describe(chol)} %  [{time.time() - t0:.0f} s, total {(time.time() - t_start) / 60:.0f} min]", flush=True)
    return nll

result = minimize(objective, theta0, method="L-BFGS-B", bounds=bounds, options={"maxiter": MAX_ITERATIONS, "eps": 1e-3})
print(f"{result.message}, {result.nit} iterations, {best['evaluations']} likelihood evaluations, {best['skipped']} trial points rejected by the step box, "
      f"{(time.time() - t_start) / 3600:.1f} h")

The estimates are reported per joint in the units of the first notebook, followed by the cross-correlations $\rho_{jk}$ between the joints at the same stride and gait point, and by the negative log likelihoods of the three nested models. The difference between two rows of the last table is the gain in log likelihood from the corresponding extension of the model.

In [ ]:
cov.set_parameters(best["theta"])
p, sigma2 = cov.parameters, best["sigma2"]
estimates = pd.DataFrame({"input range [standardized units]": p["l"], "temporal range [% gait cycle]": p["lt"],
                          "variance [BW²]": sigma2 * p["v"] * scale**2, "nugget effect [BW²]": sigma2 * p["eta"] * scale**2}, index=JOINTS).T
estimates.columns.name = "joint"
shown = estimates.apply(lambda col: col.map("{:.3f}".format))
shown.loc["nugget effect [BW²]"] = estimates.loc["nugget effect [BW²]"].map("{:.2e}".format)
shown

In [ ]:
pd.DataFrame(p["rho"], index=JOINTS, columns=JOINTS).round(3)

In [ ]:
ladder = pd.DataFrame({"parameters": [12, 11, 15], "negative log likelihood": [nll_independent, nll_equal, best["nll"]]},
                      index=["three independent univariate emulators", "trivariate, equal ranges (Kronecker)", "trivariate, joint-specific ranges (dense)"])
ladder["gain over independent"] = nll_independent - ladder["negative log likelihood"]
ladder.style.format({"negative log likelihood": "{:,.1f}", "gain over independent": "{:,.1f}"})

## References

Apanasovich, T. V., Genton, M. G. and Sun, Y. (2012). A valid Matérn class of cross-covariance functions for multivariate random fields with any number of components. *Journal of the American Statistical Association*, 107, 180-193.

Gneiting, T., Kleiber, W. and Schlather, M. (2010). Matérn cross-covariance functions for multivariate random fields. *Journal of the American Statistical Association*, 105, 1167-1177.

Higham, N. J. and Mary, T. (2022). Mixed precision algorithms in numerical linear algebra. *Acta Numerica*, 31, 347-414.

Salvaña, M. L. O. (2026). Adaptive block mixed-precision Cholesky factorization for large-scale space-time Gaussian processes.